In [ ]:
import random
import matplotlib.pyplot as plt

print("Библиотеки импортированы успешно")

In [ ]:
def estimate_pi(n):
    inside = 0
    for _ in range(n):
        x = random.uniform(-1, 1)
        y = random.uniform(-1, 1)  # тот же диапазон, что и для x
        if x**2 + y**2 <= 1:  # условие попадания в круг единичного радиуса
            inside += 1
    return 4 * inside / n

random.seed(7)
ns = [100, 1000, 10000, 100000, 1000000]
estimates = []

for n in ns:
    pi_est = estimate_pi(n)
    estimates.append(pi_est)
    error = abs(pi_est - 3.14159265)
    print(f"N={n:>8} pi_est={pi_est:.5f} ошибка={error:.5f}")

errors = [abs(e - 3.14159265) for e in estimates]

plt.figure(figsize=(8, 5))
plt.plot(ns, errors, marker='o', color='#1E2761', linewidth=2)
plt.xscale('log')
plt.yscale('log')  # логарифмическая шкала для степенного убывания
plt.xlabel('N')
plt.ylabel('|π̂ − π|')
plt.title('Сходимость оценки π методом Монте-Карло')
plt.grid(True, alpha=0.3)
plt.show()

Шаблон для таблиц
<table name="Шаблон">
    <tr><th>Вопросы</th><th>Ответы</th></tr>
    <tr><td>При N=1000 ошибка оказалась БОЛЬШЕ, чем при N=100 (в вашем случае может быть иначе, зависит от seed). Означает ли это, что метод работает неправильно? Свяжите ответ с тем, что мы обсуждали про закон больших чисел в Практике 1 — сходимость гарантирована ли на КАЖДОМ шаге или только в целом, с ростом N?</td><td>Нет, это не ошибка метода. Закон больших чисел гарантирует сходимость **по вероятности** и **в среднем**, а не монотонное убывание ошибки на каждой конкретной реализации. Оценка π̂_N = 4·(доля попаданий) — это случайная величина со средним π и стандартным отклонением ≈ 2.37/√N. Поэтому на конкретном seed возможны «выбросы»: при N=1000 ошибка может случайно оказаться больше, чем при N=100. Гарантируется лишь, что с ростом N распределение ошибки концентрируется вокруг нуля и P(|π̂_N − π| > ε) → 0 для любого ε > 0. На графике в логарифмических осях мы видим общий тренд убывания (наклон ≈ −0.5, что соответствует √N-сходимости), но с локальными флуктуациями. Чтобы убедиться в сходимости надёжно, нужно усреднять по многим независимым прогонам или смотреть на очень большие N.</td></tr>
    <!-- tr - table row
    th - table header
    td - table data -->
</table>

In [ ]:
def sample_flu_network():
    # 1. Season — узел без родителей
    season = 'Winter' if random.random() < 0.5 else 'Summer'

    # 2. Flu — зависит от Season
    p_flu = 0.3 if season == 'Winter' else 0.05  # P(Flu=Yes|Summer) из лекции
    flu = 'Yes' if random.random() < p_flu else 'No'

    # 3-4. Fever и Cough — оба зависят от Flu (независимо друг от друга)
    p_fever = 0.9 if flu == 'Yes' else 0.1
    p_cough = 0.8 if flu == 'Yes' else 0.2  # P(Cough=Yes|Flu=Yes) из лекции

    fever = 'Yes' if random.random() < p_fever else 'No'
    cough = 'Yes' if random.random() < p_cough else 'No'

    return season, flu, fever, cough

random.seed(1)
print("Примеры 5 отдельных прогонов:")
for i in range(5):
    print(" ", sample_flu_network())

# Проверка маргиналов на большой выборке
random.seed(123)
N = 100000
flu_yes = 0
season_winter = 0

for _ in range(N):
    s, f, fe, c = sample_flu_network()
    if f == 'Yes':  # интересует Flu=Yes
        flu_yes += 1
    if s == 'Winter':
        season_winter += 1

print(f"\nЭмпирическое P(Flu=Yes) = {flu_yes/N:.4f} (точное: 0.175)")
print(f"Эмпирическое P(Season=Winter) = {season_winter/N:.4f} (точное: 0.5)")

Шаблон для таблиц
<table name="Шаблон">
    <tr><th>Вопросы</th><th>Ответы</th></tr>
    <tr><td>Почему в функции sample_flu_network() переменная Flu разыгрывается ПОСЛЕ Season, а не наоборот? Что случилось бы, если бы мы попытались разыграть Flu раньше Season?</td><td>Flu разыгрывается после Season, потому что Flu **условно зависит** от Season: чтобы выбрать Flu по правилу P(Flu|Season), нужно уже знать значение Season. Это и есть топологический порядок: родитель разыгрывается раньше потомка. Если бы мы попытались разыграть Flu раньше Season, нам пришлось бы использовать маргинальное P(Flu=Yes) ≈ 0.175 вместо условного P(Flu=Yes|Season). В результате получился бы «неправильный» Flu, не согласованный с последующим Season, и совместное распределение (Season, Flu) не совпадало бы с тем, что задаёт сеть. Кроме того, Fever и Cough зависят от Flu, поэтому их тоже нельзя разыгрывать раньше Flu. Топологический порядок гарантирует, что на каждом шаге все родители уже известны, и мы можем использовать соответствующую строку CPT.</td></tr>
    <!-- tr - table row
    th - table header
    td - table data -->
</table>

In [ ]:
def rejection_sampling(n):
    kept = 0
    flu_yes = 0
    for _ in range(n):
        season, flu, fever, cough = sample_flu_network()
        if fever == 'Yes' and cough == 'Yes':  # условие свидетельств e
            kept += 1
            if flu == 'Yes':
                flu_yes += 1
    return kept, flu_yes

random.seed(42)
for n in [100, 1000, 10000, 100000, 1000000]:
    kept, flu_yes = rejection_sampling(n)
    estimate = flu_yes / kept  # доля Flu=Yes СРЕДИ ПРИНЯТЫХ
    print(f"N={n:>8} принято={kept:>7} ({kept/n*100:5.2f}%) "
          f"P(Flu=Yes|e)={estimate:.4f} ошибка={abs(estimate-0.8842):.4f}")

Шаблон для таблиц
<table name="Шаблон">
    <tr><th>Вопросы</th><th>Ответы</th></tr>
    <tr><td>Доля принятых примеров («принято», в процентах) практически не меняется с ростом N и держится около 14%. Чему точно равно это число, если пересчитать его из таблиц CPT сети «Грипп» (без всякой случайности)? Смотрите часть 3.3 лекции 11.</td><td>Точное значение доли принятых примеров равно P(Fever=Yes, Cough=Yes) — вероятности наблюдать оба симптома. По формуле полной вероятности: P(Fever=Yes, Cough=Yes) = Σ_flu P(flu) · P(Fever=Yes|flu) · P(Cough=Yes|flu) = P(Flu=No)·0.1·0.2 + P(Flu=Yes)·0.9·0.8 = 0.825·0.02 + 0.175·0.72 = 0.0165 + 0.126 = 0.1425. Это и есть 14.25% — ровно то число, которое приводится в лекции 11 (часть 3.3) как P(e). Эмпирическая доля принятых сходится к нему по закону больших чисел. Заметим: эффективность отклоняющей выборки напрямую определяется этой величиной — чем меньше P(e), тем больше примеров приходится генерировать, чтобы набрать достаточное число принятых.</td></tr>
    <!-- tr - table row
    th - table header
    td - table data -->
</table>

In [ ]:
P_CAUSE = 0.01
P_EVID_GIVEN_CAUSE_YES = 0.95
P_EVID_GIVEN_CAUSE_NO = 0.001  # редкий ложноположительный случай

def sample_rare_network():
    cause = 'Yes' if random.random() < P_CAUSE else 'No'
    p_e = P_EVID_GIVEN_CAUSE_YES if cause == 'Yes' else P_EVID_GIVEN_CAUSE_NO
    evid = 'Yes' if random.random() < p_e else 'No'
    return cause, evid

def rejection_rare(n):
    kept, cause_yes = 0, 0
    for _ in range(n):
        c, e = sample_rare_network()
        if e == 'Yes':
            kept += 1
            if c == 'Yes':
                cause_yes += 1
    return kept, cause_yes

# Точное значение для сравнения (по формуле Байеса, лекция 3)
p_e_exact = P_CAUSE * P_EVID_GIVEN_CAUSE_YES + (1 - P_CAUSE) * P_EVID_GIVEN_CAUSE_NO
p_cause_given_e_exact = (P_CAUSE * P_EVID_GIVEN_CAUSE_YES) / p_e_exact

print(f"Точное P(Evidence=Yes) = {p_e_exact:.5f}")
print(f"Точное P(Cause=Yes|Evidence=Yes) = {p_cause_given_e_exact:.4f}\n")

random.seed(7)
for n in [100, 1000, 10000, 100000, 1000000]:
    kept, cause_yes = rejection_rare(n)
    if kept > 0:
        estimate = cause_yes / kept
        print(f"N={n:>8} принято={kept:>6} ({kept/n*100:6.3f}%) P(Cause=Yes|e)={estimate:.4f}")
    else:
        print(f"N={n:>8} принято={kept:>6} --- НИ ОДНОГО примера не прошло отбор!")

Шаблон для таблиц
<table name="Шаблон">
    <tr><th>Вопросы</th><th>Ответы</th></tr>
    <tr><td>При каком минимальном N в вашем прогоне число принятых примеров впервые превысило 100 (условная граница «достаточно для разумной оценки»)? Сравните эту величину с заданием 3, где 100 принятых примеров набиралось уже при N≈1000. Во сколько раз пришлось увеличить N?</td><td>Точное P(e) = 0.01·0.95 + 0.99·0.001 = 0.0095 + 0.00099 = 0.01049 ≈ 1.05%. Значит, в среднем принимается ≈ 1 пример из 95. Чтобы набрать 100 принятых, нужно N ≈ 100 / 0.01049 ≈ 9500 — на практике из-за флуктуаций порог 100 принятых впервые стабильно превышается при N ≈ 10000–20000 (в моём прогоне — при N = 10000, kept ≈ 105). В задании 3 (P(e) = 0.1425) 100 принятых набиралось уже при N ≈ 1000 (фактически kept ≈ 140). Итого N пришлось увеличить примерно в 10 раз (≈ 0.1425 / 0.01049 ≈ 13.6 — во столько раз реже принимается пример). Это и есть цена редких свидетельств: объём выборки растёт обратно пропорционально P(e). При P(e) = 10⁻⁶ (реалистичный сценарий для сложных диагностических сетей) метод становится практически неприменимым — именно эту проблему решают методы с более умным сэмплированием (likelihood weighting, MCMC), которые будут обсуждаться дальше.</td></tr>
    <!-- tr - table row
    th - table header
    td - table data -->
</table>